## Harbor Installation

Dieses Notebook enthält die notwendigen Befehle zur Konfiguration von Harbor mittels Docker Compose.

Folgendes wurde bereites Installiert bzw. downgeloadet
* Harbor nach ~/harbor
* docker-compose
* cert-manager


### TLS Zertifikate 

* Erstellen der Zertifikate für Harbor
* Auslesen der Zertifikate aus Kubernetes

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: cert-manager.io/v1
kind: Certificate
metadata:
  name: selfsigned-cert-ch
spec:
  secretName: selfsigned-cert-secret
  duration: 2400h
  renewBefore: 12h
  issuerRef:
    name: selfsigned-cluster-issuer
    kind: ClusterIssuer
  commonName: dev.$(cat ~/work/server-ip)-edutbz.com
  dnsNames:
    - dev.$(cat ~/work/server-ip)-edutbz.com
EOF

In [ ]:
%%bash
cd ~/harbor
kubectl get secret selfsigned-cert-secret -o jsonpath="{.data.tls\.crt}" | base64 --decode > ~/harbor/tls.crt
kubectl get secret selfsigned-cert-secret -o jsonpath="{.data.tls\.key}" | base64 --decode > ~/harbor/tls.key

### Zertifikatsinhalt prüfen

In [ ]:
%%bash
cd ~/harbor
openssl x509 -in tls.crt -text -noout

### Harbor Konfiguration anpassen

In [ ]:
%%bash
cd ~/harbor
cp harbor.yml.tmpl harbor.yml
sed -i -e "s/reg.mydomain.com/dev.$(cat ~/work/server-ip)-edutbz.com/g" harbor.yml
sed -i -e 's/port: 80/port: 9090/g' harbor.yml
sed -i -e 's/port: 443/port: 9443/g' harbor.yml
sed -i -e 's;certificate: /your/certificate/path;certificate: /home/ubuntu/harbor/tls.crt;g' harbor.yml
sed -i -e 's;private_key: /your/private/key/path;private_key: /home/ubuntu/harbor/tls.key;g' harbor.yml

### Harbor starten

In [ ]:
%%bash
cd ~/harbor
sudo ./install.sh --with-trivy

Harbor ist dann via dem untenstehenden URL erreichbar.
Benutzer: `admin`, Passwort: `Harbor12345`.

In [ ]:
%%bash
echo "https://dev.$(cat ~/work/server-ip)-edutbz.com:9443"

### Docker: Selbst signiertes Zertifikat vertrauenswürdig machen

In [ ]:
%%bash
cd ~/harbor
kubectl get secret selfsigned-cert-secret -n m01-ch -o jsonpath="{.data.ca\.crt}" | base64 --decode > ca.crt

In [ ]:
%%bash
cd ~/harbor
sudo mkdir -p /etc/docker/certs.d/ch.$(cat ~/work/server-ip)-edutbz.com:9443
sudo cp ca.crt /etc/docker/certs.d/ch.$(cat ~/work/server-ip)-edutbz.com:9443/ca.crt

Docker und Harbor neu starten, damit das Zertifikat geladen wird:

In [ ]:
%%bash
cd ~/harbor
sudo docker-compose down -v
sudo systemctl restart docker
sudo ./install.sh --with-trivy

Jetzt sollte `docker login` ohne Fehlermeldung funktionieren.

    docker login dev.[hostname]-edutbz.com:9443

### Aufträge
- Repliziert Eure Container Registries bzw. die Images nach Harbor.
- Erstellt ein Container Image und legt es in die Harbor Registry ab.

**Links:**
[Harbor Installer](https://goharbor.io/docs/2.12.0/install-config/run-installer-script/)